In [2]:
import numpy as np
import torch

In [3]:
class LoRALayer(torch.nn.Module):
    def __init__(self, in_features, out_features, rank, alpha):
        super().__init__()

        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = torch.nn.Parameter(torch.randn(in_features, rank) * std_dev)
        self.B = torch.nn.Parameter(torch.zeros(rank, out_features))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

In [7]:
lora = LoRALayer(26, 26, 4, .1)
lora.A

Parameter containing:
tensor([[-0.4904, -0.2422,  0.6496,  0.0214],
        [-0.4571, -0.4529, -0.0447,  0.1423],
        [ 0.0973,  0.2993, -1.1225,  0.7741],
        [-0.0422,  1.1705, -0.9898, -0.1267],
        [ 0.7113,  0.0063,  0.3302,  0.1339],
        [-0.1602, -0.0578, -0.2328,  0.6969],
        [-0.1601, -0.0110,  0.1870, -0.0034],
        [-0.3209, -0.6645,  0.0389, -0.4904],
        [ 0.0226, -0.3454, -0.0888, -0.2531],
        [ 0.1480, -0.0133, -0.2399, -0.7764],
        [ 0.1907, -0.0681,  0.2510,  0.4488],
        [ 0.3343, -0.4341,  0.5650, -0.9453],
        [ 0.0552, -0.7238, -0.4018, -0.4215],
        [ 0.0723, -0.1809, -0.5936, -0.5204],
        [ 0.3797,  0.7728, -0.0868, -0.2018],
        [-0.1005,  0.0793, -0.3358,  0.0548],
        [-0.3100,  0.4319, -0.1273, -0.6622],
        [ 0.2090,  0.1404,  0.5765, -0.7077],
        [-0.7476, -0.7704, -0.4112,  0.0266],
        [-0.1436, -0.5389,  0.2547, -0.0610],
        [ 0.1281, -0.4669, -0.6306,  0.0072],
        [-0.

In [8]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [ ]:
linear = torch.nn.Linear(8, 4)
lora_linear = LinearWithLoRA(linear, rank=2, alpha=0.1)

x = torch.randn(3,8)


tensor([[-0.1136, -1.3150,  0.2367,  0.0883, -1.1894, -0.2270,  0.6566, -1.0515],
        [-1.0726,  0.2094,  0.3648,  0.1994,  0.4805,  2.0931, -0.5310, -1.6212],
        [-0.8100, -2.0671, -2.1132, -0.2158,  0.3879,  0.4659, -1.3530,  0.2156]])

In [18]:
lora_linear(x)

tensor([[ 0.2471, -0.1814, -0.6511,  1.0608],
        [ 0.5192,  1.0204, -0.2834,  0.0554],
        [-0.1214, -1.3311, -1.7106,  0.9966]], grad_fn=<AddBackward0>)